In [0]:
pip install faker

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from faker import Faker
import random

In [0]:
fake=Faker("en_US")
random.seed(42)


### Generate Customers

In [0]:
num_customers=100000
customer_data=[]
for i in range(1,num_customers+1):
    customer_data.append(
        (i,fake.name(),
         fake.city(),
         fake.state(),
         "USA"))

#### Schema

In [0]:
customer_schema=StructType([StructField('customer_id',IntegerType(),False),
                            StructField('customer_name',StringType(),False),
                            StructField('city',StringType(),False),
                            StructField('state',StringType(),False),
                            StructField('country',StringType(),False)])

#### Create DataFrame

In [0]:
customers_df=spark.createDataFrame(customer_data,schema=customer_schema)
customers_df.show(5)

+-----------+----------------+------------------+-------------+-------+
|customer_id|   customer_name|              city|        state|country|
+-----------+----------------+------------------+-------------+-------+
|          1|   Jimmy Francis|        Wilsonport|         Iowa|    USA|
|          2|    Karen Wilson|North Jeremiahstad|        Maine|    USA|
|          3|   Tony Mcdonald|        Gordonstad|    Wisconsin|    USA|
|          4| Danielle Brooks|     North Melissa|     Missouri|    USA|
|          5|Meredith Stevens|       Bushborough|West Virginia|    USA|
+-----------+----------------+------------------+-------------+-------+
only showing top 5 rows


#### Write to volume

In [0]:
customers_df.write.mode('overwrite').parquet('/Volumes/workspace/spark_optimization/project_data/customers')

### Generate Products

In [0]:
product_catalog={
    'electronics':['Wireless Mouse','Keyboard','Monitor','Headphones','Laptop'],
    'clothing':['T-Shirt','Pants','Shirt','Dress','Jacket'],
    'home_appliances':['Refrigerator','Washing Machine','Microwave','Toaster'],
    'books':['Fiction','Non-fiction','Biography','Textbook','Poetry'],
    'sports':['Basketball','Football','Tennis']
}
num_products=5000
product_data=[]
for i in range(1,num_products+1):
  category=random.choice(list(product_catalog.keys()))
  product_name=random.choice(product_catalog[category])

  product_data.append(
      (i,
       product_name,
       category,
       random.randint(1,1000)
      )
  )


#### Schema


In [0]:
product_schema=StructType([StructField('product_id',IntegerType(),False),
                            StructField('product_name',StringType(),False),
                            StructField('category',StringType(),False),
                            StructField('price',IntegerType(),False)])

#### Create DataFrame

In [0]:
product_df=spark.createDataFrame(product_data,product_schema)
product_df.show(5)

+----------+---------------+---------------+-----+
|product_id|   product_name|       category|price|
+----------+---------------+---------------+-----+
|         1|Washing Machine|home_appliances|  687|
|         2|   Refrigerator|home_appliances|  624|
|         3|         Jacket|       clothing|  747|
|         4|          Pants|       clothing|  474|
|         5|      Biography|          books|  948|
+----------+---------------+---------------+-----+
only showing top 5 rows


#### Write to Volume

In [0]:
product_df.write.mode('overwrite').parquet('/Volumes/workspace/spark_optimization/project_data/products')

### Generate Orders

In [0]:
num_orders=1000000
orders_df=(spark.range(1,num_orders+1)
        .withColumnRenamed('id','order_id')
        .withColumn('customer_id',(floor(rand(42)*100000)+1).cast('int'))
        .withColumn('product_id',(floor(rand(42)*5000)+1).cast('int'))
        .withColumn('quantity',(floor(rand(456)*5)+1).cast('int'))
        .withColumn('order_date',expr("date_add('2023-02-02',cast(rand(789)*1095 as int))")))
       
       

#### Add price

In [0]:
orders_df=(orders_df.join(product_df.select('product_id','price'),on='product_id',how='left'))

#### Add Total Amount

In [0]:
orders_df=orders_df.withColumn('total_amount',col('price')*col('quantity'))

#### Create Partition Columns 

In [0]:
orders_df=orders_df.withColumn('order_year',year('order_date'))
orders_df=orders_df.withColumn('order_month',month('order_date'))
orders_df.show(5)
orders_df.count()

+----------+--------+-----------+--------+----------+-----+------------+----------+-----------+
|product_id|order_id|customer_id|quantity|order_date|price|total_amount|order_year|order_month|
+----------+--------+-----------+--------+----------+-----+------------+----------+-----------+
|       429|       1|       8576|       4|2023-08-03|  969|        3876|      2023|          8|
|      1553|       2|      31042|       5|2024-08-24|  718|        3590|      2024|          8|
|       313|       3|       6257|       1|2023-12-13|  619|         619|      2023|         12|
|      1533|       4|      30647|       3|2025-08-06|  746|        2238|      2025|          8|
|         1|       5|          5|       1|2024-07-29|  687|         687|      2024|          7|
+----------+--------+-----------+--------+----------+-----+------------+----------+-----------+
only showing top 5 rows


1000000

#### Write to Volume

In [0]:
orders_df.write.mode('overwrite').partitionBy('order_year','order_month').parquet('/Volumes/workspace/spark_optimization/project_data/orders')

In [0]:
import shutil

shutil.make_archive(
    "/tmp/project_data",
    "zip",
    "/Volumes/workspace/spark_optimization/project_data"
)

'/tmp/project_data.zip'

In [0]:
import os

print(os.path.exists("/tmp/project_data.zip"))
print(os.listdir("/tmp")[:20])

True
['hsperfdata_root', 'databricks-jni13169407015392064924', 'custom-spark.conf', 'tmpdja1hetx', 'start_pre_checkpoint', 'backup_process_log', 'temp9463109245921708478snapstart-jni.so', 'dbr.conf', 'python_lsp_logs', 'chauffeur-env.sh', 'driver-daemon.pid', 'dbr_entry_point.py', 'backup_bind_mount', 'keyutil_spark.host.local_6057952746377074400.crt', 'chauffeur-daemon-params', 'keyutil_spark.host.local_3475175717954237636.key', 'matplotlib-root', 'temp2503747602710346221snapstart-jni.so', 'tmpy474np7y.py', 'pre_start.sh']


In [0]:
import shutil

shutil.copy("/tmp/project_data.zip", "/Workspace/Users/niranjanags1201@gmail.com/project_data.zip")

'/Workspace/Users/niranjanags1201@gmail.com/project_data.zip'